In [17]:
import json
import yaml

In [1]:
env_content = """
HUGGINGFACE_API_KEY=your_actual_api_key_here
"""

with open(".env", "w") as f:
    f.write(env_content.strip())

print("✅ .env 파일이 생성되었습니다.")

✅ .env 파일이 생성되었습니다.


In [13]:
def write_config(model_name, config_data):
    # json 파일로 저장
    number = model_name.split("_")[-1]
    config_filename = f"config_{number}.json"
    with open(f'models/{model_name}/{config_filename}', 'w', encoding='utf-8') as f:
        json.dump(config_data, f, indent=4)
    
    print(f"✅ models/{model_name}/{config_filename} 저장 완료!")

In [27]:
def write_yaml(model_name, yaml_data):
    number = model_name.split("_")[-1]
    yaml_filename = f"yaml_{number}.yaml"
    with open(f'models/{model_name}/{yaml_filename}', 'w', encoding='utf-8') as f:
        yaml.dump(yaml_data, f, sort_keys=False)
    print(f'✅ models/{model_name}/{yaml_filename} 저장 완료!')

In [22]:
config_data = {
    "model_name": "yolo11m-seg.pt",
    "project_name": "alphadent_yolo_runs",
    "run_name": "alphadent_model_355",

    "epochs": 200,
    "patience": 20,
    "image_size": 1536,
    "batch_size": 6,
    "conf_thres": 0.01,
    "iou_thres": 0.45,

    "data": "/models/model_355/yaml_355.yaml",
    "device": 0,
    "workers": 8,
    "project": '/models/model_355',
    "exist_ok": True,
    "seed": 42,
    "amp": True,          

    # --------------------------------------------------------------------------
    # 2. Optimizer & Learning Rate
    # --------------------------------------------------------------------------
    "optimizer": "AdamW",
    "lr0": 1e-4,
    "lrf": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "warmup_epochs": 5.0,
    "cos_lr": True,

    # --------------------------------------------------------------------------
    # 3. Augmentation (치아 데이터 맞춤)
    # --------------------------------------------------------------------------
    "hsv_h": 0.015,
    "hsv_s": 0.4,
    "hsv_v": 0.4,
    "degrees": 10.0,
    "translate": 0.01,
    "scale": 0.2,
    "fliplr": 0.5,
    "flipud": 0.15,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0,
    "close_mosaic": 10,

    # --------------------------------------------------------------------------
    # 4. Segmentation & Validation 수정 (에러 해결 핵심!)
    # --------------------------------------------------------------------------
    "overlap_mask": True,
    "retina_masks": True,
    "mask_ratio": 2,
    "box": 7.5,
    "cls": 1.5,
    "dfl": 1.5,
    "val": True,             # 에포크마다 검증 수행
    "plots": True,            # 학습 결과 시각화 저장

    "caf_alpha": 0.5,
    "nwd_alpha": 0.6,
    
}

model_name = "model_355"

write_config(model_name, config_data)

✅ models/model_355/config_355.json 저장 완료!


In [32]:
model_yaml_data = {
    'path': '/data/alphadent_roi',  # 데이터셋 루트 경로 (절대 경로 추천)
    'train': 'images/train',                    # 학습 이미지 경로 (path 기준 상대 경로)
    'val': 'images/valid',                      # 검증 이미지 경로
    'test': 'images/test',                      # 테스트 이미지 경로 (옵션)
    'nc': 9,
    'names': {0: 'Abrasion', 1: 'Filling', 2: 'Crown',
              3: 'Caries Class 1', 4: 'Caries Class 2', 5: 'Caries Class 3',
              6: 'Caries Class 4', 7: 'Caries Class 5', 8: 'Caries Class 6'},
    
    'scales': {'m': [0.50, 1.00, 512]},

    # ============================================================
    # Backbone
    # ============================================================
    'backbone': [
        [-1, 1, 'Conv', [64, 3, 2]],          # [0] P1/2
        [-1, 1, 'Conv', [128, 3, 2]],         # [1] P2/4
        [-1, 2, 'C3k2', [256, False, 0.25]],  # [2]

        [-1, 1, 'Conv', [256, 3, 2]],         # [3] P3/8
        [-1, 2, 'C3k2', [512, False, 0.25]],  # [4]

        [-1, 1, 'Conv', [512, 3, 2]],         # [5] P4/16
        [-1, 2, 'C3k2', [512, True]],         # [6]

        [-1, 1, 'Conv', [1024, 3, 2]],        # [7] P5/32
        [-1, 2, 'C3k2', [1024, True]],        # [8]

        [-1, 1, 'SPPF', [1024, 5]],           # [9]
        [-1, 2, 'C2PSA', [1024]],             # [10]
    ],

    # ============================================================
    # Head (CAF only at P3 + P4)
    # ============================================================
    'head': [

        # ---------- Top-down FPN ----------
        [-1, 1, 'nn.Upsample', [None, 2, "nearest"]],  # [11]
        [[-1, 6], 1, 'Concat', [1]],                   # [12]
        [-1, 2, 'C3k2', [512, False]],                 # [13] fused P4

        [-1, 1, 'nn.Upsample', [None, 2, "nearest"]],  # [14]
        [[-1, 4], 1, 'Concat', [1]],                   # [15]
        [-1, 2, 'C3k2', [256, False]],                 # [16] P3 (small)

        [-1, 1, 'CAFBlock', [256]],                    # [17] ⭐ CAF @ P3

        # ---------- Bottom-up PAN ----------
        [-1, 1, 'Conv', [256, 3, 2]],                  # [18]
        [[-1, 13], 1, 'Concat', [1]],                  # [19]
        [-1, 2, 'C3k2', [512, False]],                 # [20] refined P4

        [-1, 1, 'CAFBlock', [512]],                    # [21] ⭐ CAF @ P4

        [-1, 1, 'Conv', [512, 3, 2]],                  # [22]
        [[-1, 10], 1, 'Concat', [1]],                  # [23]
        [-1, 2, 'C3k2', [1024, True]],                 # [24] P5

        # ---------- Segmentation Head ----------
        [[17, 21, 24], 1, 'Segment', [9, 32, 256]],    # [25]
    ]
}

write_yaml("model_355", model_yaml_data)

✅ models/model_355/yaml_355.yaml 저장 완료!


In [33]:
config_data = {
    "model_name": "yolo11m-seg.pt",
    "project_name": "alphadent_yolo_runs",
    "run_name": "alphadent_model_357",

    "epochs": 200,
    "patience": 20,
    "image_size": 1536,
    "batch_size": 4,
    "conf_thres": 0.01,
    "iou_thres": 0.45,

    "data": "/models/model_357/yaml_357.yaml",
    "device": 0,
    "workers": 8,
    "project": '/models/model_357',
    "exist_ok": True,
    "seed": 42,
    "amp": True,          

    # --------------------------------------------------------------------------
    # 2. Optimizer & Learning Rate
    # --------------------------------------------------------------------------
    "optimizer": "AdamW",
    "lr0": 1e-4,
    "lrf": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "warmup_epochs": 5.0,
    "cos_lr": True,

    # --------------------------------------------------------------------------
    # 3. Augmentation (치아 데이터 맞춤)
    # --------------------------------------------------------------------------
    "hsv_h": 0.015,
    "hsv_s": 0.4,
    "hsv_v": 0.4,
    "degrees": 10.0,
    "translate": 0.01,
    "scale": 0.2,
    "fliplr": 0.5,
    "flipud": 0.2,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0,
    "close_mosaic": 10,

    # --------------------------------------------------------------------------
    # 4. Segmentation & Validation 수정 (에러 해결 핵심!)
    # --------------------------------------------------------------------------
    "overlap_mask": True,
    "retina_masks": True,
    "mask_ratio": 2,
    "box": 7.5,
    "cls": 1.5,
    "dfl": 1.5,
    "val": True,             # 에포크마다 검증 수행
    "plots": True,            # 학습 결과 시각화 저장

    "caf_alpha": 0.5,
    "nwd_alpha": 0.6,
    
}

model_name = "model_357"

write_config(model_name, config_data)

✅ models/model_357/config_357.json 저장 완료!


In [34]:
model_yaml_data = {
    'path': '/data/alphadent_roi',  # 데이터셋 루트 경로 (절대 경로 추천)
    'train': 'images/train',                    # 학습 이미지 경로 (path 기준 상대 경로)
    'val': 'images/valid',                      # 검증 이미지 경로
    'test': 'images/test',                      # 테스트 이미지 경로 (옵션)
    'nc': 9,
    'names': {0: 'Abrasion', 1: 'Filling', 2: 'Crown',
              3: 'Caries Class 1', 4: 'Caries Class 2', 5: 'Caries Class 3',
              6: 'Caries Class 4', 7: 'Caries Class 5', 8: 'Caries Class 6'}
}

write_yaml("model_357", model_yaml_data)

✅ models/model_357/yaml_357.yaml 저장 완료!


In [35]:
config_data = {
    "model_name": "yolo11m-seg.pt",
    "project_name": "alphadent_yolo_runs",
    "run_name": "alphadent_model_360",

    "epochs": 200,
    "patience": 20,
    "image_size": 768,
    "batch_size": 40,
    "conf_thres": 0.01,
    "iou_thres": 0.45,

    "data": "/models/model_360/yaml_360.yaml",
    "device": 0,
    "workers": 8,
    "project": '/models/model_360',
    "exist_ok": True,
    "seed": 42,       

    # --------------------------------------------------------------------------
    # 2. Optimizer & Learning Rate
    # --------------------------------------------------------------------------
    "optimizer": "AdamW",
    "lr0": 1e-4,
    "lrf": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "warmup_epochs": 5.0,
    "cos_lr": True,

    # --------------------------------------------------------------------------
    # 3. Augmentation (치아 데이터 맞춤)
    # --------------------------------------------------------------------------
    "hsv_h": 0.015,
    "hsv_s": 0.4,
    "hsv_v": 0.4,
    "degrees": 10.0,
    "translate": 0.1,
    "scale": 0.2,
    "fliplr": 0.5,
    "flipud": 0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0,
    "close_mosaic": 10,

    # --------------------------------------------------------------------------
    # 4. Segmentation & Validation 수정 (에러 해결 핵심!)
    # --------------------------------------------------------------------------
    "overlap_mask": True,
    "retina_masks": True,
    "mask_ratio": 2,
    "box": 7.5,
    "cls": 1.5,
    "dfl": 1.5,
    "val": True,             # 에포크마다 검증 수행
    "plots": True,            # 학습 결과 시각화 저장

    "caf_alpha": 0.5,
    "nwd_alpha": 0.6,
    
}

model_name = "model_360"

write_config(model_name, config_data)

✅ models/model_360/config_360.json 저장 완료!


In [36]:
model_yaml_data = {
    'path': '/data/alphadent_instance',  # 데이터셋 루트 경로 (절대 경로 추천)
    'train': 'images/train',                    # 학습 이미지 경로 (path 기준 상대 경로)
    'val': 'images/valid',                      # 검증 이미지 경로
    'test': 'images/test',                      # 테스트 이미지 경로 (옵션)
    'nc': 9,
    'names': {0: 'Abrasion', 1: 'Filling', 2: 'Crown',
              3: 'Caries Class 1', 4: 'Caries Class 2', 5: 'Caries Class 3',
              6: 'Caries Class 4', 7: 'Caries Class 5', 8: 'Caries Class 6'}
}

write_yaml("model_360", model_yaml_data)

✅ models/model_360/yaml_360.yaml 저장 완료!


In [37]:
config_data = {
    "model_name": "yolo11m-seg.pt",
    "project_name": "alphadent_yolo_runs",
    "run_name": "alphadent_model_365",

    "epochs": 200,
    "patience": 30,
    "image_size": 1536,
    "batch_size": 6,
    "conf_thres": 0.01,
    "iou_thres": 0.45,

    "data": "/models/model_365/yaml_365.yaml",
    "device": 0,
    "workers": 8,
    "project": '/models/model_365',
    "exist_ok": True,
    "seed": 42,       

    # --------------------------------------------------------------------------
    # 2. Optimizer & Learning Rate
    # --------------------------------------------------------------------------
    "optimizer": "AdamW",
    "lr0": 1e-4,
    "lrf": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "warmup_epochs": 5.0,
    "cos_lr": True,

    # --------------------------------------------------------------------------
    # 3. Augmentation (치아 데이터 맞춤)
    # --------------------------------------------------------------------------
    "hsv_h": 0.015,
    "hsv_s": 0.4,
    "hsv_v": 0.4,
    "degrees": 10.0,
    "translate": 0.01,
    "scale": 0.2,
    "fliplr": 0.5,
    "flipud": 0.15,
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0,
    "close_mosaic": 10,

    # --------------------------------------------------------------------------
    # 4. Segmentation & Validation 수정 (에러 해결 핵심!)
    # --------------------------------------------------------------------------
    "overlap_mask": True,
    "retina_masks": True,
    "mask_ratio": 2,
    "box": 7.5,
    "cls": 1.5,
    "dfl": 1.5,
    "val": True,             # 에포크마다 검증 수행
    "plots": True,            # 학습 결과 시각화 저장

    "caf_alpha": 0.5,
    "nwd_alpha": 0.6,
    
}

model_name = "model_365"

write_config(model_name, config_data)

✅ models/model_365/config_365.json 저장 완료!


In [38]:
model_yaml_data = {
    'path': '/data/alphadent_roi',  # 데이터셋 루트 경로 (절대 경로 추천)
    'train': 'images/train',                    # 학습 이미지 경로 (path 기준 상대 경로)
    'val': 'images/valid',                      # 검증 이미지 경로
    'test': 'images/test',                      # 테스트 이미지 경로 (옵션)
    'nc': 9,
    'names': {0: 'Abrasion', 1: 'Filling', 2: 'Crown',
              3: 'Caries Class 1', 4: 'Caries Class 2', 5: 'Caries Class 3',
              6: 'Caries Class 4', 7: 'Caries Class 5', 8: 'Caries Class 6'},
    
    'scales': {'m': [0.50, 1.00, 512]},

    # ============================================================
    # Backbone (기존과 동일)
    # ============================================================
    'backbone': [
        [-1, 1, 'Conv', [64, 3, 2]],          # [0] P1/2
        [-1, 1, 'Conv', [128, 3, 2]],         # [1] P2/4
        [-1, 2, 'C3k2', [256, False, 0.25]],  # [2]
        [-1, 1, 'Conv', [256, 3, 2]],         # [3] P3/8
        [-1, 2, 'C3k2', [512, False, 0.25]],  # [4]
        [-1, 1, 'Conv', [512, 3, 2]],         # [5] P4/16
        [-1, 2, 'C3k2', [512, True]],         # [6]
        [-1, 1, 'Conv', [1024, 3, 2]],        # [7] P5/32
        [-1, 2, 'C3k2', [1024, True]],        # [8]
        [-1, 1, 'SPPF', [1024, 5]],           # [9]
        [-1, 2, 'C2PSA', [1024]],             # [10]
    ],

    # ============================================================
    # Head: Triple CAF (P3, P4, P5)
    # ============================================================
    'head': [
        # ---------- Top-down FPN ----------
        [-1, 1, 'nn.Upsample', [None, 2, "nearest"]],  # [11]
        [[-1, 6], 1, 'Concat', [1]],                   # [12]
        [-1, 2, 'C3k2', [512, False]],                 # [13] fused P4

        [-1, 1, 'nn.Upsample', [None, 2, "nearest"]],  # [14]
        [[-1, 4], 1, 'Concat', [1]],                   # [15]
        [-1, 2, 'C3k2', [256, False]],                 # [16] P3 Initial
        [-1, 1, 'CAFBlock', [256]],                    # [17] ⭐ CAF @ P3 (강도 조절 적용)

        # ---------- Bottom-up PAN ----------
        [-1, 1, 'Conv', [256, 3, 2]],                  # [18]
        [[-1, 13], 1, 'Concat', [1]],                  # [19]
        [-1, 2, 'C3k2', [512, False]],                 # [20] P4 Initial
        [-1, 1, 'CAFBlock', [512]],                    # [21] ⭐ CAF @ P4 (강도 조절 적용)

        [-1, 1, 'Conv', [512, 3, 2]],                  # [22]
        [[-1, 10], 1, 'Concat', [1]],                  # [23]
        [-1, 2, 'C3k2', [1024, True]],                 # [24] P5 Initial
        [-1, 1, 'CAFBlock', [512]],                    # [25] ⭐ CAF @ P5 (강도 조절 적용)

        # ---------- Segmentation Head ----------
        # 정제된 모든 레벨의 특징 [17, 21, 25] 사용
        [[17, 21, 25], 1, 'Segment', [9, 32, 256]],    # [26]
    ]
}

write_yaml("model_365", model_yaml_data)

✅ models/model_365/yaml_365.yaml 저장 완료!
